In [1]:
import json
import pandas as pd
import numpy as np
import os
from chronos import Chronos2Pipeline
from sklearn.metrics import root_mean_squared_error

path = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
with open(path, "r") as f:
    dataset_days = json.load(f)

countries = ["Germany","Ireland","Portugal"]
days = ["day1","day2","day3","day4","day5"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation"
]

pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

rmse_results = []

for country in countries:
    print("Processing country:", country)

    data_path = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = [col for col in df.columns if col not in features]

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []  # reset per day

        for household in households:
            # training slice
            s_train = df.loc[df.index < cutoff, household]
            X_train = df.loc[s_train.index, features]

            df_household_train = pd.concat([s_train.rename(household), X_train], axis=1)
            df_household_train_last = df_household_train.tail(10000).reset_index()
            df_household_train_last["id"] = household

            pred_df = pipeline.predict_df(
                df_household_train_last,
                prediction_length=96,
                id_column="id",
                timestamp_column="timestamp",
                target=household,
            )

            # init predictions df with correct timestamp index (once)
            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=pred_df["timestamp"])

            # store predictions
            predictions_df_all_households[household] = pred_df["predictions"].to_numpy()

            # RMSE for this household (aligned by timestamp)
            y_pred = predictions_df_all_households[household]
            y_true = df.loc[y_pred.index, household]

            rmse = root_mean_squared_error(y_true, y_pred)  # already RMSE
            rmse_households.append(rmse)

        avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households
        })

        # save predictions for this (country, day)
        output = rf"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2Univar_pred_{day}_{country.capitalize()}.csv"
        os.makedirs(os.path.dirname(output), exist_ok=True)
        predictions_df_all_households.to_csv(output, index=True)
        print("      Saved:", output)

# summary table
rmse_df = pd.DataFrame(rmse_results)
print("\nPer-day RMSE:")
print(rmse_df)

print("\nCross-validated RMSE per country (mean over days):")
print(rmse_df.groupby("country")["rmse"].mean())

c:\Users\CR58XM\AppData\Local\anaconda3\envs\chronos\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing country: Germany
   Day: day1


c:\Users\CR58XM\AppData\Local\anaconda3\envs\chronos\Lib\site-packages\chronos\chronos2\dataset.py:89: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_numpy.cpp:212.)
  task_target = torch.from_numpy(task_target)


      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2Univar_pred_day1_Germany.csv
   Day: day2
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2Univar_pred_day2_Germany.csv
   Day: day3
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2Univar_pred_day3_Germany.csv
   Day: day4
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2Univar_pred_day4_Germany.csv
   Day: day5
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2Univar_pred_day5_Germany.csv
Processing country: Ireland
   Day: day1
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\Chronos2Univar_pred_day1_Ireland.csv
   Day: day2
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\